# Qwen3 0.6B + Mimi Stage 1 Training With Unsloth

This notebook keeps the useful parts of the Unsloth training flow:

```text
FastLanguageModel.from_pretrained
FastLanguageModel.get_peft_model
LoRA / Unsloth-optimized Qwen backbone
Trainer-based training loop
adapter saving
```

But the training objective is no longer text-only SFT. It is a Stage 1 speech-code objective for the dataset shaped like:

```text
id: string
text: string
speaker_id: int32
codes: [8, n_frames]
n_frames: int32
k_codebooks: 8
```

The model is a hybrid:

```text
Unsloth Qwen3 0.6B LoRA backbone
+ previous-Mimi-frame embeddings
+ 8 parallel Mimi codebook heads
```

At each Mimi frame, the Qwen temporal backbone sees one frame position and the heads predict all 8 codebooks. This is the real-time-friendly architecture direction. It avoids making Qwen autoregress through 8 flattened codebook tokens per frame.

## What This Notebook Is For

This is Stage 1 only:

```text
transcript text + previous Mimi frames -> next Mimi frame cb0-cb7
```

Expected after Stage 1:

```text
loss goes down
Mimi code predictions become non-collapsed
short generated code sequences can be decoded later with Mimi
```

Not expected after Stage 1:

```text
full duplex behavior
clean turn-taking
interruptions/backchannels
end-of-speech control
```

Those come later in Stage 2 and Stage 3.

## Install

This follows the original Unsloth notebook pattern. It does not force-upgrade Torch, because GPU VMs often already have a CUDA-matched PyTorch install.

In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets>=2.20.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install -U "datasets>=2.20.0" "accelerate>=0.33.0" "transformers>=4.56.0" "safetensors>=0.4.3"

## Configuration

The dataset default matches the sample row you showed, for example `1272-128104-0000` and lower-case LibriSpeech text.

If you later want punctuation-preserved TTS-style text, change `DATASET_NAME` to `shangeth/libritts-r-mimi-codes`.

In [ ]:
import math
import os
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments, set_seed
from unsloth import FastLanguageModel

set_seed(3407)

MODEL_NAME = "Qwen/Qwen3-0.6B"
DATASET_NAME = "shangeth/librispeech-mimi-codes"
SPLIT = "train_clean_100"
OUTPUT_DIR = Path("qwen3_0_6b_mimi_stage1_unsloth")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_CODEBOOKS = 8
CODEBOOK_SIZE = 2048
START_CODE = CODEBOOK_SIZE  # input-only code used before the first frame

MAX_TEXT_TOKENS = 192
MAX_FRAMES = 192            # 192 frames ~= 15.36 sec at 12.5 fps
MAX_SEQ_LENGTH = MAX_TEXT_TOKENS + MAX_FRAMES + 8

# Start small. Increase only after the smoke run works.
TRAIN_LIMIT = 2_000         # None for full split
EVAL_LIMIT = 128
MAX_STEPS = 1_000
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4

# Qwen 0.6B is small. bf16 LoRA is usually safer for this custom wrapper than 4-bit.
# Set True if VRAM is tight and the VM supports bitsandbytes well.
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
LORA_R = 32
LORA_ALPHA = 32
CB0_LOSS_WEIGHT = 8.0

print("model:", MODEL_NAME)
print("dataset:", DATASET_NAME, SPLIT)
print("max_seq_length:", MAX_SEQ_LENGTH)
print("output:", OUTPUT_DIR)

## Load Qwen With Unsloth

This is the main Unsloth part. We load Qwen through `FastLanguageModel`, then attach LoRA adapters with `get_peft_model`.

For this custom audio-frame trainer, we do not use `SFTTrainer`, because we are not training `input_ids -> labels`. We train:

```text
text tokens + previous Mimi frames -> current Mimi frame codebooks
```

In [ ]:
qwen_lm, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=LOAD_IN_8BIT,
    full_finetuning=False,
    # token="YOUR_HF_TOKEN",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

qwen_lm = FastLanguageModel.get_peft_model(
    qwen_lm,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
qwen_lm.config.use_cache = False

print("tokenizer vocab:", len(tokenizer))
print("pad token:", tokenizer.pad_token)
print("loaded in 4bit:", getattr(qwen_lm, "is_loaded_in_4bit", False))

## Load Mimi-Code Dataset

Expected columns:

```text
id, text, speaker_id, codes, n_frames, k_codebooks
```

The collator truncates long samples to `MAX_FRAMES` for the first run.

In [ ]:
raw = load_dataset(DATASET_NAME, split=SPLIT)
print(raw)
print(raw.column_names)
print("first id:", raw[0]["id"])
print("first text:", raw[0]["text"])
print("first n_frames:", raw[0]["n_frames"], "k:", raw[0]["k_codebooks"])

if TRAIN_LIMIT is not None:
    raw = raw.select(range(min(len(raw), TRAIN_LIMIT + EVAL_LIMIT)))

split = raw.train_test_split(test_size=min(EVAL_LIMIT, max(1, len(raw) // 20)), seed=3407)
train_raw = split["train"]
eval_raw = split["test"]

print("train rows:", len(train_raw))
print("eval rows:", len(eval_raw))

## Dataset And Collator

The collator returns tensors for a custom model:

```text
input_ids: [batch, text_len]
attention_mask: [batch, text_len]
codes: [batch, 8, frames]
code_mask: [batch, frames]
```

The model internally shifts `codes` so previous frames predict current frames.

In [ ]:
class MimiCodeRows(Dataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[idx]
        return {
            "id": row["id"],
            "text": row["text"],
            "speaker_id": row["speaker_id"],
            "codes": row["codes"],
            "n_frames": row["n_frames"],
            "k_codebooks": row["k_codebooks"],
        }


@dataclass
class MimiFrameCollator:
    tokenizer: object
    max_text_tokens: int = MAX_TEXT_TOKENS
    max_frames: int = MAX_FRAMES
    k_codebooks: int = K_CODEBOOKS

    def __call__(self, rows):
        prompts = [self._format_prompt(row["text"]) for row in rows]
        text_batch = self.tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=self.max_text_tokens,
            return_tensors="pt",
            add_special_tokens=True,
        )

        frame_count = min(
            self.max_frames,
            max(min(int(row["n_frames"]), len(row["codes"][0])) for row in rows),
        )
        codes = torch.full(
            (len(rows), self.k_codebooks, frame_count),
            fill_value=-100,
            dtype=torch.long,
        )
        code_mask = torch.zeros((len(rows), frame_count), dtype=torch.long)

        for batch_idx, row in enumerate(rows):
            if int(row["k_codebooks"]) < self.k_codebooks:
                raise ValueError(f"row {row['id']} has only {row['k_codebooks']} codebooks")
            row_codes = torch.tensor(row["codes"][: self.k_codebooks], dtype=torch.long)
            if row_codes.ndim != 2 or row_codes.shape[0] != self.k_codebooks:
                raise ValueError(f"row {row['id']} codes must be [8, n_frames], got {tuple(row_codes.shape)}")
            if row_codes.min() < 0 or row_codes.max() >= CODEBOOK_SIZE:
                raise ValueError(f"row {row['id']} has Mimi code outside 0-{CODEBOOK_SIZE - 1}")
            n = min(frame_count, row_codes.shape[1], self.max_frames)
            codes[batch_idx, :, :n] = row_codes[:, :n]
            code_mask[batch_idx, :n] = 1

        return {
            "input_ids": text_batch["input_ids"],
            "attention_mask": text_batch["attention_mask"],
            "codes": codes,
            "code_mask": code_mask,
        }

    @staticmethod
    def _format_prompt(text: str) -> str:
        return f"<speech_text> {text.strip()}\n<audio_codes>"


train_dataset = MimiCodeRows(train_raw)
eval_dataset = MimiCodeRows(eval_raw)
collator = MimiFrameCollator(tokenizer)

batch = collator([train_dataset[0], train_dataset[1]])
for key, value in batch.items():
    print(key, tuple(value.shape), value.dtype)

## Temporal Qwen + Mimi Codebook Heads

This wrapper keeps the Unsloth LoRA model as the temporal backbone.

For each frame position:

```text
input embedding = frame marker + embeddings(previous cb0-cb7)
Qwen hidden state = temporal context
8 heads = logits for current cb0-cb7
```

`cb0` gets higher loss weight because it carries the strongest speech-content signal.

In [ ]:
def get_underlying_lm(model):
    if hasattr(model, "get_base_model"):
        return model.get_base_model()
    return model


def get_qwen_transformer(model):
    base = get_underlying_lm(model)
    if hasattr(base, "model") and hasattr(base.model, "layers"):
        return base.model
    if hasattr(base, "base_model") and hasattr(base.base_model, "model"):
        candidate = base.base_model.model
        if hasattr(candidate, "layers"):
            return candidate
    raise TypeError("Could not locate Qwen transformer backbone on the loaded model")


def first_model_device(model):
    return next(model.parameters()).device


class UnslothQwenMimiFrameModel(nn.Module):
    def __init__(
        self,
        qwen_lm,
        k_codebooks: int = K_CODEBOOKS,
        codebook_size: int = CODEBOOK_SIZE,
        start_code: int = START_CODE,
        cb0_weight: float = CB0_LOSS_WEIGHT,
    ):
        super().__init__()
        self.qwen_lm = qwen_lm
        self.config = qwen_lm.config
        self.k_codebooks = k_codebooks
        self.codebook_size = codebook_size
        self.start_code = start_code
        hidden_size = qwen_lm.config.hidden_size

        self.frame_marker = nn.Parameter(torch.zeros(hidden_size))
        self.codebook_embeddings = nn.ModuleList(
            [nn.Embedding(codebook_size + 1, hidden_size) for _ in range(k_codebooks)]
        )
        self.codebook_heads = nn.ModuleList(
            [nn.Linear(hidden_size, codebook_size) for _ in range(k_codebooks)]
        )
        weights = torch.ones(k_codebooks)
        weights[0] = cb0_weight
        self.register_buffer("codebook_loss_weights", weights, persistent=False)
        self._init_audio_modules()
        self.move_audio_modules_to(first_model_device(qwen_lm))

        # Preserve quantization flags so HF Trainer does not treat the wrapper as a plain fp32 model.
        for attr in ["is_loaded_in_4bit", "is_loaded_in_8bit", "hf_device_map"]:
            if hasattr(qwen_lm, attr):
                setattr(self, attr, getattr(qwen_lm, attr))

    def _init_audio_modules(self):
        nn.init.normal_(self.frame_marker, mean=0.0, std=0.02)
        for emb in self.codebook_embeddings:
            nn.init.normal_(emb.weight, mean=0.0, std=0.02)
        for head in self.codebook_heads:
            nn.init.normal_(head.weight, mean=0.0, std=0.02)
            nn.init.zeros_(head.bias)

    def move_audio_modules_to(self, device):
        self.codebook_embeddings.to(device)
        self.codebook_heads.to(device)
        self.frame_marker.data = self.frame_marker.data.to(device)
        self.codebook_loss_weights.data = self.codebook_loss_weights.data.to(device)

    def qwen_transformer(self):
        return get_qwen_transformer(self.qwen_lm)

    def text_embeddings(self):
        return self.qwen_lm.get_input_embeddings()

    def build_prev_codes(self, codes: torch.Tensor) -> torch.Tensor:
        safe_codes = codes.clamp(min=0)
        prev_codes = torch.full_like(safe_codes, fill_value=self.start_code)
        prev_codes[:, :, 1:] = safe_codes[:, :, :-1]
        prev_codes = torch.where(codes.eq(-100), torch.full_like(prev_codes, self.start_code), prev_codes)
        return prev_codes

    def frame_inputs_from_prev_codes(self, prev_codes: torch.Tensor) -> torch.Tensor:
        frame_marker = self.frame_marker.view(1, 1, -1).to(prev_codes.device)
        pieces = []
        for cb in range(self.k_codebooks):
            pieces.append(self.codebook_embeddings[cb](prev_codes[:, cb, :]))
        stacked = torch.stack(pieces, dim=0).sum(dim=0) / math.sqrt(self.k_codebooks)
        return stacked + frame_marker

    def forward_logits(self, input_ids, attention_mask, prev_codes, code_mask):
        text_emb = self.text_embeddings()(input_ids)
        frame_emb = self.frame_inputs_from_prev_codes(prev_codes).to(dtype=text_emb.dtype)
        inputs_embeds = torch.cat([text_emb, frame_emb], dim=1)
        full_attention_mask = torch.cat([attention_mask, code_mask], dim=1)
        outputs = self.qwen_transformer()(inputs_embeds=inputs_embeds, attention_mask=full_attention_mask, use_cache=False)
        audio_hidden = outputs.last_hidden_state[:, input_ids.shape[1] :, :]
        return [head(audio_hidden.to(dtype=head.weight.dtype)) for head in self.codebook_heads]

    def forward(self, input_ids, attention_mask, codes, code_mask):
        codes = codes.to(first_model_device(self.qwen_lm))
        code_mask = code_mask.to(first_model_device(self.qwen_lm))
        input_ids = input_ids.to(first_model_device(self.qwen_lm))
        attention_mask = attention_mask.to(first_model_device(self.qwen_lm))
        prev_codes = self.build_prev_codes(codes)
        logits_by_cb = self.forward_logits(input_ids, attention_mask, prev_codes, code_mask)

        total_loss = None
        total_weight = 0.0
        per_codebook_losses = []
        for cb, logits in enumerate(logits_by_cb):
            labels = codes[:, cb, :].contiguous()
            loss = F.cross_entropy(
                logits.reshape(-1, self.codebook_size).float(),
                labels.reshape(-1),
                ignore_index=-100,
            )
            per_codebook_losses.append(loss.detach())
            weight = self.codebook_loss_weights[cb].float()
            total_loss = loss * weight if total_loss is None else total_loss + loss * weight
            total_weight += float(weight.item())
        total_loss = total_loss / total_weight
        return {"loss": total_loss, "logits": logits_by_cb[0]}

    def audio_state_dict(self):
        prefixes = ("frame_marker", "codebook_embeddings", "codebook_heads")
        return {k: v.detach().cpu() for k, v in self.state_dict().items() if k.startswith(prefixes)}


model = UnslothQwenMimiFrameModel(qwen_lm)
qwen_lm = None

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total params: {total/1e6:.1f}M")
print(f"trainable params: {trainable/1e6:.1f}M ({trainable/total:.2%})")
print("device:", first_model_device(model.qwen_lm))

## Forward Smoke Test

Run one batch through the model before starting the full run. If this cell fails, fix that before training.

In [ ]:
smoke_batch = collator([train_dataset[0], train_dataset[1]])
model.train()
smoke_out = model(**smoke_batch)
print("smoke loss:", float(smoke_out["loss"].detach().cpu()))
print("cb0 logits:", tuple(smoke_out["logits"].shape))

## Train

This uses the Hugging Face `Trainer`, not `SFTTrainer`, because the model has a custom forward pass and custom audio-codebook loss.

Checkpoint saving is disabled during training to avoid accidentally writing the full wrapped Qwen state every few steps. The final cell saves:

```text
qwen_lora/              LoRA adapter from Unsloth/PEFT
mimi_frame_heads.pt     frame marker, Mimi embeddings, and 8 codebook heads
metadata.json           training metadata
```

In [ ]:
use_cuda = torch.cuda.is_available()
bf16_ok = bool(use_cuda and torch.cuda.is_bf16_supported())

if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16:", bf16_ok)
else:
    print("CUDA not available. This notebook is intended for GPU training.")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,
    logging_steps=10,
    # Unsloth patches Trainer evaluation for prompt-style batches.
    # Disable built-in eval here; use the manual eval cell below for this custom audio batch.
    eval_strategy="no",
    save_strategy="no",
    bf16=bf16_ok and not LOAD_IN_4BIT and not LOAD_IN_8BIT,
    fp16=bool(use_cuda and not bf16_ok and not LOAD_IN_4BIT and not LOAD_IN_8BIT),
    optim="adamw_8bit" if use_cuda else "adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    gradient_checkpointing=False,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=None,
    data_collator=collator,
)

trainer.train()

## Manual Evaluation

Unsloth patches the default `Trainer.evaluate()` path for prompt-style text/RL batches. This custom Mimi batch does not have a `prompt` field, so use this manual eval loop instead.


In [ ]:
@torch.no_grad()
def manual_eval_loss(model, dataset, collator, max_batches: int = 32):
    model.eval()
    losses = []
    for start in range(0, min(len(dataset), max_batches * BATCH_SIZE), BATCH_SIZE):
        rows = [dataset[i] for i in range(start, min(start + BATCH_SIZE, len(dataset)))]
        batch = collator(rows)
        out = model(**batch)
        losses.append(float(out["loss"].detach().cpu()))
    model.train()
    return sum(losses) / max(1, len(losses))

print("manual eval loss:", manual_eval_loss(model, eval_dataset, collator))

## Save Adapter And Audio Heads

This saves the LoRA adapter separately from the custom Mimi modules. That is safer than saving a full custom wrapped model.

In [ ]:
import json

final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(parents=True, exist_ok=True)

model.qwen_lm.save_pretrained(final_dir / "qwen_lora")
tokenizer.save_pretrained(final_dir / "tokenizer")
torch.save(model.audio_state_dict(), final_dir / "mimi_frame_heads.pt")

metadata = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "split": SPLIT,
    "k_codebooks": K_CODEBOOKS,
    "codebook_size": CODEBOOK_SIZE,
    "start_code": START_CODE,
    "max_text_tokens": MAX_TEXT_TOKENS,
    "max_frames": MAX_FRAMES,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "cb0_loss_weight": CB0_LOSS_WEIGHT,
    "load_in_4bit": LOAD_IN_4BIT,
    "load_in_8bit": LOAD_IN_8BIT,
}
(final_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
print("saved:", final_dir)

## Generate Mimi Codes From Text

This generates code IDs only. To hear audio, pass the resulting `[8, frames]` codes into a Mimi decoder.

Stage 1 has no learned stop/silence behavior yet, so generation uses a fixed `max_frames`. Stage 2/3 will add silence/speak control.

In [ ]:
@torch.no_grad()
def generate_mimi_codes(model, tokenizer, text: str, max_frames: int = 80, temperature: float = 0.8):
    model.eval()
    device = first_model_device(model.qwen_lm)
    prompt = MimiFrameCollator._format_prompt(text)
    encoded = tokenizer(
        [prompt],
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_TOKENS,
        return_tensors="pt",
        add_special_tokens=True,
    )
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    prev_codes = torch.full((1, K_CODEBOOKS, 1), START_CODE, dtype=torch.long, device=device)
    generated = []

    for _ in range(max_frames):
        code_mask = torch.ones((1, prev_codes.shape[-1]), dtype=torch.long, device=device)
        logits_by_cb = model.forward_logits(input_ids, attention_mask, prev_codes, code_mask)
        next_codes = []
        for cb in range(K_CODEBOOKS):
            logits = logits_by_cb[cb][:, -1, :] / max(temperature, 1e-6)
            probs = torch.softmax(logits.float(), dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(1)
            next_codes.append(next_id)
        next_frame = torch.stack(next_codes, dim=1)  # [1, 8]
        generated.append(next_frame.squeeze(0).detach().cpu())
        prev_codes = torch.cat([prev_codes, next_frame.unsqueeze(-1)], dim=-1)

    return torch.stack(generated, dim=1)  # [8, frames]

sample_codes = generate_mimi_codes(
    model,
    tokenizer,
    "mister quilter is the apostle of the middle classes and we are glad to welcome his gospel",
    max_frames=40,
)
print(sample_codes.shape)
print(sample_codes[:, :5])
print("unique cb0 tokens:", len(set(sample_codes[0].tolist())))

## Expected Result

Good first-run signs:

```text
smoke forward pass works
training loss decreases
eval loss decreases or at least does not explode
sampled code IDs are diverse, not one repeated token
```

Known limitations:

```text
This is Stage 1 only.
Generated codes need a Mimi decoder to become audio.
There is no end-of-speech token yet.
This does not teach turn-taking or full duplex.
```

Next implementation steps:

```text
1. Add Mimi decode smoke test.
2. Move this model/trainer into repo Python scripts.
3. Train Stage 1 beyond the 2K-row debug subset.
4. Add Stage 2 clean dialogue frame-stream training.
5. Add Stage 3 overlap/full-duplex training.
```